# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [2]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [3]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


Número de estados: 500
Número de acciones: 6


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.
Esto es debido a que se toman en cuenta las 25 posiciones del taxi, las 5 ubicaciones del pasajero y los 4 destinos posibles, es decir, 25x5x4 = 500. En cambio en forzen lake solo se toman las 16 posiciones del agente 

## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [4]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


Estado: 386
Acción: 3
Nuevo estado: 366
Recompensa: -1
Terminated: False


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?
$$
s_t = 386(estado en el que estaba),\quad a_t = 3(accion que tomo),\quad r_{t+1} = -1(recompensa),\quad s_{t+1} = 366(nuevo estado)
$$

## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$

**Respuesta:**

Taxi-v4 tiene 500 estados y 6 acciones posibles. Por lo tanto, el agente debe aprender:

$$
500 \times 6 = 3000
$$

En total, la Q-table contiene 3000 valores. Cada uno indica qué tan conveniente es realizar una acción determinada desde un estado específico.

## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [ ]:
def choose_action(Q, state, epsilon, env):
    # Exploración: con probabilidad epsilon se prueba una acción al azar.
    if random.random() < epsilon:
        return env.action_space.sample()

    # Explotación: se toma argmax_a Q(s, a).
    q_values = Q[state]
    max_q = np.max(q_values)

    # Desempate aleatorio: al inicio toda la fila vale 0 y argmax
    # devolvería siempre la acción 0 (South).
    best_actions = np.flatnonzero(q_values == max_q)
    return int(np.random.choice(best_actions))


## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [ ]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    # Target: recompensa inmediata + valor descontado del mejor futuro.
    best_next_q = np.max(Q[next_state])
    td_target = reward + gamma * best_next_q

    # Error TD: cuánto se equivocó la estimación actual.
    td_error = td_target - Q[state, action]

    # Se corrige la estimación una fracción alpha del error.
    Q[state, action] += alpha * td_error

    return Q


## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):

            # 1. Escoger acción con la política epsilon-greedy.
            action = choose_action(Q, state, epsilon, env)

            # 2. Ejecutar la acción en el ambiente.
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 3. Actualizar Q(s, a) con la regla de Q-Learning.
            update_q(Q, state, action, reward, next_state, alpha, gamma)

            # 4. Avanzar al nuevo estado y acumular la recompensa.
            state = next_state
            total_reward += reward

            # 5. Terminar el episodio si el ambiente lo indica.
            if done:
                break

        rewards.append(total_reward)

    return Q, rewards


## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?

**Respuesta:**

La recompensa promedio debería mejorar con el paso de los episodios. Al principio suele ser baja porque el taxi todavía se mueve de manera poco eficiente y comete errores. Después de unos cientos o algunos miles de episodios, la curva normalmente empieza a estabilizarse, aunque puede presentar pequeñas variaciones porque todavía existe un poco de exploración.

El resultado no demuestra una convergencia perfecta. Más bien indica que el agente aprendió una política razonablemente buena: en general sabe recoger y transportar al pasajero, pero todavía puede tardar más de lo ideal o cometer algún movimiento innecesario.

## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
- ¿El taxi sin entrenamiento logra completar la tarea?
- ¿El agente entrenado evita acciones innecesarias?
- ¿Qué evidencia visual le permite afirmar que el agente aprendió?

**Respuesta:**

El taxi sin entrenamiento se mueve sin una estrategia clara. Puede chocar repetidamente con las paredes, alejarse del pasajero o quedarse sin tiempo antes de completar la tarea. Por eso, normalmente no logra llevar al pasajero a su destino.

Después del entrenamiento, el taxi sigue una ruta más organizada: se acerca al pasajero, lo recoge y después se dirige hacia el destino. También reduce los movimientos innecesarios, aunque no siempre encuentra la ruta absolutamente más corta.

La evidencia visual es que el taxi entrenado completa la tarea con mayor frecuencia y realiza acciones relacionadas con el objetivo. Además, su recompensa suele ser mejor que la del episodio sin entrenamiento.

## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?

**Respuesta:**

1. La acción con mayor valor Q es la que aparece en `Mejor acción` al ejecutar la celda. En Taxi, los códigos son: 0 = South, 1 = North, 2 = East, 3 = West, 4 = Pickup y 5 = Dropoff.

2. Que una acción tenga un Q mayor significa que, según lo aprendido, esa acción ofrece una mejor posibilidad de completar la tarea y obtener buenas recompensas en el futuro.

3. No representa solamente la recompensa inmediata. También considera lo que puede ocurrir después de esa acción: los siguientes movimientos, las posibles penalizaciones y la recompensa final por llevar correctamente al pasajero a su destino.

## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.

**Respuesta:**

Al aumentar $\epsilon$, el agente explora más y prueba acciones nuevas con mayor frecuencia. Esto puede hacer que aprenda más lentamente al principio y que la recompensa sea más variable, pero también puede ayudarlo a descubrir mejores rutas. Al reducir $\epsilon$, aprovecha más lo que ya aprendió y su comportamiento suele ser más estable, aunque corre el riesgo de quedarse con una estrategia que no sea la mejor.

En general, el entrenamiento original ofrece un buen punto de comparación: la recompensa sube gradualmente y el taxi termina siguiendo rutas útiles. El nuevo entrenamiento debe evaluarse observando si aprende antes, si la curva tiene más variaciones y si consigue una recompensa final mayor. Una recompensa más alta y una curva más estable indicarían una mejora; una curva muy irregular o una recompensa menor indicarían que el cambio no fue conveniente.

## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
